# 11. Causal ML in Industry

## Background

Academic causal inference asks "what is the effect?" Industry causal ML asks "what should we *do* with that estimate?" The gap between identification and action is bridged by three applied problems: **A/B analysis** (translating experimental results into decisions), **uplift modeling** (predicting who benefits from treatment and how much), and **policy learning** (finding the optimal treatment rule under constraints).

These aren't just academic extensions. At scale, the value of CATE over ATE is enormous: a company that can identify the 20% of users who benefit most from a feature and target only them may achieve 80% of the benefit at 20% of the cost.

## What You'll Learn

- Cumulative gain curves and Qini coefficient: the uplift evaluation framework
- S-learner, T-learner, and DR-learner for uplift
- Ranking users by predicted CATE for targeted deployment
- Policy trees: interpretable treatment assignment rules
- Expected value of a policy: translating CATE into business impact

## Why This Matters

The `causalml` library (Uber, 2019) and `econml` (Microsoft) provide production-grade uplift tools. The Qini curve is the standard metric for uplift models (Radcliffe 2007). Connecting CATE estimates to revenue or cost savings is how causal inference earns a seat at the product table.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.tree import DecisionTreeRegressor, export_text
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
def simulate_uplift_study(n=4000, seed=0):
    rng = np.random.default_rng(seed)
    recency   = rng.exponential(30, n)
    frequency = rng.poisson(3, n)
    monetary  = rng.lognormal(4, 1, n)
    age       = rng.normal(40, 12, n)
    mobile    = rng.binomial(1, 0.6, n)
    # True CATE: recent frequent buyers benefit most
    tau_x = (0.05 + 0.02 * np.clip(frequency, 0, 5)
              - 0.001 * recency + 0.01 * mobile
              - 0.003 * np.abs(age - 35))
    tau_x = np.clip(tau_x, -0.05, 0.20)
    ps = np.clip(0.3 + 0.02*frequency - 0.002*recency, 0.1, 0.9)
    T  = rng.binomial(1, ps)
    p0 = np.clip(0.05 + 0.01*frequency - 0.001*recency + 0.01*mobile, 0.01, 0.5)
    p1 = np.clip(p0 + tau_x, 0.01, 0.99)
    Y  = np.where(T, rng.binomial(1, p1), rng.binomial(1, p0))
    return pd.DataFrame({'recency':recency,'frequency':frequency,'monetary':monetary,
                         'age':age,'mobile':mobile,'T':T,'Y':Y,'tau_true':tau_x})


df = simulate_uplift_study(n=5000)
feat_cols = ['recency','frequency','monetary','age','mobile']
print(f"n={len(df)}, treated={df.T.sum()}, converted={df.Y.sum()}")
print(f"True ATE = {df.tau_true.mean():.4f}, CATE range: [{df.tau_true.min():.3f}, {df.tau_true.max():.3f}]")

## 1. T-Learner and DR-Learner Uplift Models

In [ ]:
X_tr, X_te, T_tr, T_te, Y_tr, Y_te, tau_tr, tau_te = train_test_split(
    df[feat_cols].values, df.T.values, df.Y.values, df.tau_true.values,
    test_size=0.3, random_state=0)

def t_learner(X_tr, T_tr, Y_tr, X_te):
    m1 = GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=0)
    m0 = GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=0)
    m1.fit(X_tr[T_tr==1], Y_tr[T_tr==1])
    m0.fit(X_tr[T_tr==0], Y_tr[T_tr==0])
    return m1.predict_proba(X_te)[:,1] - m0.predict_proba(X_te)[:,1]


def dr_learner_binary(X_tr, T_tr, Y_tr, X_te, n_folds=5, seed=0):
    n = len(Y_tr)
    mu1 = np.zeros(n); mu0 = np.zeros(n); ps = np.zeros(n)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(X_tr):
        m1 = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=seed)
        m0 = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=seed)
        mp = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=seed)
        Xf, Xv = X_tr[tr_idx], X_tr[val_idx]
        Tf, Yf = T_tr[tr_idx], Y_tr[tr_idx]
        m1.fit(Xf[Tf==1], Yf[Tf==1]); m0.fit(Xf[Tf==0], Yf[Tf==0]); mp.fit(Xf, Tf)
        mu1[val_idx] = m1.predict_proba(Xv)[:,1]
        mu0[val_idx] = m0.predict_proba(Xv)[:,1]
        ps[val_idx]  = np.clip(mp.predict_proba(Xv)[:,1], 0.05, 0.95)
    gamma = (mu1 - mu0 + T_tr*(Y_tr-mu1)/ps - (1-T_tr)*(Y_tr-mu0)/(1-ps))
    m_cate = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=0)
    m_cate.fit(X_tr, gamma)
    return m_cate.predict(X_te)


tau_hat_t  = t_learner(X_tr, T_tr, Y_tr, X_te)
tau_hat_dr = dr_learner_binary(X_tr, T_tr, Y_tr, X_te)
print(f"T-learner  RMSE: {np.sqrt(np.mean((tau_hat_t  - tau_te)**2)):.4f}")
print(f"DR-learner RMSE: {np.sqrt(np.mean((tau_hat_dr - tau_te)**2)):.4f}")

## 2. Qini Curve — Uplift Evaluation

In [ ]:
def qini_curve(tau_true, tau_hat, T, Y, n_bins=100):
    df_eval = pd.DataFrame({'tau_true':tau_true,'tau_hat':tau_hat,'T':T,'Y':Y})
    df_model  = df_eval.sort_values('tau_hat',  ascending=False).reset_index(drop=True)
    df_oracle = df_eval.sort_values('tau_true', ascending=False).reset_index(drop=True)
    fracs = np.linspace(0, 1, n_bins+1)
    n = len(df_eval)
    model_q = [0]; oracle_q = [0]
    for frac in fracs[1:]:
        k = max(1, int(frac * n))
        for q_list, df_s in [(model_q, df_model), (oracle_q, df_oracle)]:
            sub = df_s.iloc[:k]
            st = sub[sub.T==1]; sc = sub[sub.T==0]
            up = (st.Y.sum()/len(st) - sc.Y.sum()/len(sc)) * frac * n if len(st)>0 and len(sc)>0 else 0
            q_list.append(up)
    scale = max(max(model_q), max(oracle_q), 1)
    mq = np.array(model_q)/scale; oq = np.array(oracle_q)/scale
    rand = np.linspace(0, mq[-1], n_bins+1)
    return fracs, mq, oq, float(np.trapz(mq-rand, fracs)), float(np.trapz(oq-rand, fracs))


fracs, qini_t, qini_oracle, q_t, q_oracle = qini_curve(tau_te, tau_hat_t,  T_te, Y_te)
fracs, qini_dr, _,           q_dr, _      = qini_curve(tau_te, tau_hat_dr, T_te, Y_te)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(fracs, qini_oracle,'k--', lw=2, label=f'Oracle (Q={q_oracle:.3f})')
ax.plot(fracs, qini_dr, color='#F44336', lw=2, label=f'DR-learner (Q={q_dr:.3f})')
ax.plot(fracs, qini_t,  color='#2196F3', lw=2, label=f'T-learner (Q={q_t:.3f})')
ax.plot(fracs, np.linspace(0, qini_oracle[-1], len(fracs)), 'gray', ls=':', lw=1.5, label='Random')
ax.set_xlabel('Fraction targeted'); ax.set_ylabel('Normalized cumulative uplift')
ax.set_title('Qini Curve — Uplift Model Comparison')
ax.legend()
plt.tight_layout()
plt.savefig('11_qini.png', dpi=110, bbox_inches='tight')
plt.show()

## 3. Business Impact

In [ ]:
def business_impact(tau_hat, tau_true, revenue=50, cost=2):
    thresholds = np.linspace(-0.05, 0.20, 60)
    rows = []
    for thresh in thresholds:
        targeted = tau_hat >= thresh
        profit = tau_true[targeted].sum() * revenue - targeted.sum() * cost
        rows.append({'threshold':thresh,'profit':profit,'pct_targeted':targeted.mean()})
    return pd.DataFrame(rows)


biz = business_impact(tau_hat_dr, tau_te)
best = biz.loc[biz.profit.idxmax()]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(biz.pct_targeted * 100, biz.profit, color='#4CAF50', lw=2)
ax.axvline(best.pct_targeted*100, color='red', ls='--',
           label=f'Optimal: target {best.pct_targeted:.0%}')
ax.set_xlabel('% of users targeted'); ax.set_ylabel('Expected profit ($)')
ax.set_title(f'Business Impact — Optimal profit ${best.profit:.0f} at {best.pct_targeted:.0%} coverage')
ax.legend()
plt.tight_layout()
plt.savefig('11_business_impact.png', dpi=110, bbox_inches='tight')
plt.show()
print(f"Optimal threshold: CATE > {best.threshold:.3f}")
print(f"Target {best.pct_targeted:.1%} of users for max profit ${best.profit:.0f}")

## 4. Interpretable Policy Tree

In [ ]:
# Fit shallow tree on DR pseudo-outcomes for interpretable policy
gamma_full = dr_learner_binary(X_tr, T_tr, Y_tr, X_tr)  # in-sample pseudo-outcomes
tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=50, random_state=0)
tree.fit(X_tr, gamma_full)

print("Policy Tree (treat units in leaves with CATE > treatment cost):")
print(export_text(tree, feature_names=feat_cols, max_depth=3))

# Leaf CATE summary
leaf_ids = tree.apply(X_te)
leaf_df = pd.DataFrame({'leaf':leaf_ids, 'tau_hat':tau_hat_dr}).groupby('leaf').agg(
    n=('tau_hat','count'), mean_cate=('tau_hat','mean')
).sort_values('mean_cate', ascending=False)
print("\nLeaf CATE summary (deploy: treat leaves with mean_cate > cost/revenue):")
print(leaf_df.to_string())